In [1]:
# ЭТАП 1. Загрузка данных и подготовка матриц
import time
from pathlib import Path

import numpy as np
import pandas as pd

# --- корень репозитория (работает, если ноутбук лежит в notebooks/ или в корне) ---
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# --- где лежит base_grid_clean.csv (проверяем две папки) ---
CANDIDATES = [
    ROOT / "data" / "processed" / "base_grid_clean.csv",
    ROOT / "data" / "raw" / "base_grid_clean.csv",
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Не найден base_grid_clean.csv. Положи его в data/processed/")

HORIZON = 64   # горизонт прогноза, дней
t0 = time.time()

# --- чтение с экономными типами (иначе файл займёт в разы больше памяти) ---
dtypes = {
    "warehouse_name": "category", "article": "category",
    "stock": "float32", "quantity": "float32",
    "cities_name": "category",
    "category_lvl_1": "category", "category_lvl_2": "category", "category_lvl_3": "category",
}
df = pd.read_csv(DATA_PATH, dtype=dtypes, parse_dates=["date"])
df = df.sort_values(["warehouse_name", "article", "date"]).reset_index(drop=True)

# --- номер серии (пара склад × артикул) ---
KEYS = ["warehouse_name", "article"]
df["sid"] = df.groupby(KEYS, observed=True).ngroup()
S = df["sid"].nunique()

# --- календарь: все даты подряд от первой до последней ---
dates = pd.date_range(df["date"].min(), df["date"].max())
D = len(dates)
day_idx = (df["date"] - dates[0]).dt.days.values

# --- матрицы «серия × день»: Q = продажи, ST = остатки (NaN = данных нет) ---
Q = np.full((S, D), np.nan, dtype=np.float32)
Q[df["sid"].values, day_idx] = df["quantity"].values
ST = np.full((S, D), np.nan, dtype=np.float32)
ST[df["sid"].values, day_idx] = df["stock"].values

Q0 = np.nan_to_num(Q)                       # продажи, где NaN заменён нулём
first_idx = np.argmax(~np.isnan(Q), axis=1)  # индекс первого дня серии (возраст ряда)

# --- справочник серий: склад и категория (категория известна не во всех строках) ---
meta = df.drop_duplicates("sid").set_index("sid").sort_index()
art_cat = (
    df[df["category_lvl_1"] != "unknown"]
    .drop_duplicates("article")
    .set_index("article")["category_lvl_1"]
)
meta["cat1"] = meta["article"].map(art_cat).astype("object").fillna("unknown")
wh_code = meta["warehouse_name"].cat.codes.values
cat_code = pd.Categorical(meta["cat1"]).codes

del df  # освобождаем память

print(f"Файл: {DATA_PATH}")
print(f"Серий: {S}, дней: {D}, период: {dates[0].date()} — {dates[-1].date()}")
print(f"Прогноз пойдёт на: {(dates[-1] + pd.Timedelta(days=1)).date()} — "
      f"{(dates[-1] + pd.Timedelta(days=HORIZON)).date()}")
print(f"Доля нулевых продаж: {(Q0 == 0).mean():.1%}, доля пропусков остатка: {np.isnan(ST).mean():.1%}")
print(f"Готово за {time.time() - t0:.1f} c")

Файл: C:\Users\nikit\sber-forecast\data\processed\base_grid_clean.csv
Серий: 8789, дней: 962, период: 2023-12-13 — 2026-07-31
Прогноз пойдёт на: 2026-08-01 — 2026-10-03
Доля нулевых продаж: 88.5%, доля пропусков остатка: 42.0%
Готово за 25.0 c


In [2]:
# ЭТАП 2. Метрики жюри и базовые прогнозы на нескольких окнах
def compute_metrics(Y, P):
    """Y, P — матрицы (серии × дни). Логика совпадает с score.py из ТЗ."""
    P = np.clip(P, 0, None)

    # SMAPE: по каждой серии считаем среднее по дням, где факт или прогноз не нулевой
    den = (np.abs(Y) + np.abs(P)) / 2
    with np.errstate(divide="ignore", invalid="ignore"):
        err = np.where(den > 0, np.abs(Y - P) / den, 0.0)
    cnt = (den > 0).sum(axis=1)
    smape = np.where(cnt > 0, err.sum(axis=1) / np.maximum(cnt, 1), 0.0).mean() * 100

    # WAPE, RMSE, RMSLE: по всем точкам сразу
    wape = np.abs(Y - P).sum() / np.abs(Y).sum() * 100
    rmse = np.sqrt(((Y - P) ** 2).mean())
    rmsle = np.sqrt(((np.log1p(P) - np.log1p(Y)) ** 2).mean())

    # |Bias|: по каждой серии, серии с нулевым средним фактом не считаем
    my, mp = Y.mean(axis=1), P.mean(axis=1)
    ok = my > 0
    bias = (np.abs(mp[ok] - my[ok]) / my[ok]).mean() * 100

    return {"SMAPE": smape, "WAPE": wape, "RMSE": rmse, "Bias": bias, "RMSLE": rmsle}


def window_cut(k):
    """Индекс последнего известного дня для окна проверки номер k (0 — самое свежее)."""
    return D - 1 - HORIZON * (k + 1)


def truth(c):
    """Факт за 64 дня после дня c."""
    return Q0[:, c + 1:c + 1 + HORIZON]


def flat(rate):
    """Размножаем одно число на серию в плоский прогноз на 64 дня."""
    return np.repeat(np.asarray(rate, dtype=np.float32)[:, None], HORIZON, axis=1)


def baselines(c):
    """Набор простых прогнозов на основе данных до дня c включительно."""
    out = {
        "нули": flat(np.zeros(S)),
        "последнее значение": flat(Q0[:, c]),
    }
    for w in (7, 28, 90):
        out[f"среднее {w} дн."] = flat(Q0[:, c - w + 1:c + 1].mean(axis=1))
    out["медиана 28 дн."] = flat(np.median(Q0[:, c - 27:c + 1], axis=1))
    m90 = Q0[:, c - 89:c + 1].mean(axis=1)
    out["среднее 90 дн., ноль если < 0.1"] = flat(np.where(m90 < 0.1, 0, m90))
    return out


# --- считаем таблицу по трём непересекающимся окнам и усредняем ---
N_WINDOWS = 3
rows = []
for k in range(N_WINDOWS):
    c = window_cut(k)
    Y = truth(c)
    for name, P in baselines(c).items():
        m = compute_metrics(Y, P)
        rows.append({"окно": k, "до даты": dates[c].date(), "подход": name, **m})

res = pd.DataFrame(rows)
summary = (res.groupby("подход")[["SMAPE", "WAPE", "RMSE", "Bias", "RMSLE"]]
           .mean().astype(float).round(2))
print("Окна проверки (последний известный день → прогноз на 64 дня):")
for k in range(N_WINDOWS):
    c = window_cut(k)
    print(f"  окно {k}: до {dates[c].date()}, факт {dates[c + 1].date()} — {dates[c + HORIZON].date()}, "
          f"серий без единой продажи: {(truth(c).sum(axis=1) == 0).sum()}")
print("\nСредние метрики по окнам:")
display(summary)

Окна проверки (последний известный день → прогноз на 64 дня):
  окно 0: до 2026-05-28, факт 2026-05-29 — 2026-07-31, серий без единой продажи: 1037
  окно 1: до 2026-03-25, факт 2026-03-26 — 2026-05-28, серий без единой продажи: 986
  окно 2: до 2026-01-20, факт 2026-01-21 — 2026-03-25, серий без единой продажи: 843

Средние метрики по окнам:


,SMAPE,WAPE,RMSE,Bias,RMSLE
подход,,,,,
медиана 28 дн.,168.12,76.65,0.83,92.65,0.33
нули,178.26,100.00,1.24,100.00,0.44
последнее значение,167.41,110.14,1.03,180.85,0.41
среднее 28 дн.,168.96,94.77,0.78,79.25,0.30
среднее 7 дн.,164.44,98.68,0.83,124.73,0.33
среднее 90 дн.,176.72,96.49,0.78,64.23,0.30
"среднее 90 дн., ноль если < 0.1",161.74,91.44,0.79,77.58,0.30


In [3]:
# ЭТАП 3.1. Признаки (окна не длиннее 180 дней) и сборка обучающей выборки
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


def build_features(c):
    """
    Признаки для всех серий на дату отсечки c (индекс последнего известного дня).
    Используются только данные до дня c включительно и окна не длиннее 180 дней.
    """
    f = {}
    # средние продажи по окнам разной длины
    for w in (7, 14, 28, 56, 90, 180):
        f[f"mean_{w}"] = Q0[:, c - w + 1:c + 1].mean(axis=1)
    # доля дней с продажами и максимум за окно
    for w in (28, 90, 180):
        f[f"share_pos_{w}"] = (Q0[:, c - w + 1:c + 1] > 0).mean(axis=1)
    for w in (90, 180):
        f[f"max_{w}"] = Q0[:, c - w + 1:c + 1].max(axis=1)

    # сколько дней прошло с последней продажи (180 = не было продаж за окно)
    pos = Q0[:, c - 179:c + 1] > 0
    f["days_since_sale"] = np.where(pos.any(axis=1), 179 - np.argmax(pos[:, ::-1], axis=1), 180)

    # остатки: последнее известное значение, доля дней без товара, доля пропусков
    s = ST[:, c - 179:c + 1]
    f["stock_last"] = s[:, -1]
    f["stock_missing_last"] = np.isnan(s[:, -1]).astype(np.float32)
    for w in (30, 90):
        sw = s[:, -w:]
        zero_share = np.where(np.isnan(sw), np.nan, (sw == 0).astype(np.float32))
        f[f"oos_share_{w}"] = np.nanmean(zero_share, axis=1)
    f["stock_missing_90"] = np.isnan(s[:, -90:]).mean(axis=1)
    f["stock_mean_90"] = np.nanmean(s[:, -90:], axis=1)

    # средние продажи только в дни, когда товар был (остаток предыдущего дня > 0)
    q90 = Q0[:, c - 89:c + 1]
    in_stock = ST[:, c - 90:c] > 0
    cnt_in = in_stock.sum(axis=1)
    f["mean_in_stock_90"] = np.where(cnt_in > 0, (q90 * in_stock).sum(axis=1) / np.maximum(cnt_in, 1), np.nan)

    # возраст ряда (не больше 180), склад и категория
    f["age"] = np.minimum(c - first_idx, 180)
    f["warehouse"] = wh_code
    f["category"] = cat_code

    X = pd.DataFrame(f)
    X["warehouse"] = X["warehouse"].astype("category")
    X["category"] = X["category"].astype("category")
    return X


def make_training_set(last_cut, step=7, min_age=28):
    """
    Обучающая выборка: много дат отсечки cc с шагом step дней.
    Цель — средние продажи в день за 64 дня после cc. Берём только cc <= last_cut,
    чтобы цель целиком лежала в известном прошлом.
    """
    parts_X, parts_y = [], []
    for cc in range(last_cut, 179, -step):
        X = build_features(cc)
        y = Q0[:, cc + 1:cc + 1 + HORIZON].mean(axis=1)
        keep = (cc - first_idx) >= min_age          # пропускаем слишком молодые ряды
        parts_X.append(X[keep])
        parts_y.append(y[keep])
    X_all = pd.concat(parts_X, ignore_index=True)
    for col in ("warehouse", "category"):
        X_all[col] = X_all[col].astype("category")
    return X_all, np.concatenate(parts_y)


# быстрая проверка: на последней дате признаки считаются без ошибок
t0 = time.time()
X_check = build_features(D - 1)
print(f"Признаков: {X_check.shape[1]}, серий: {X_check.shape[0]}, время {time.time() - t0:.1f} c")
print(list(X_check.columns))

Признаков: 22, серий: 8789, время 0.1 c
['mean_7', 'mean_14', 'mean_28', 'mean_56', 'mean_90', 'mean_180', 'share_pos_28', 'share_pos_90', 'share_pos_180', 'max_90', 'max_180', 'days_since_sale', 'stock_last', 'stock_missing_last', 'oos_share_30', 'oos_share_90', 'stock_missing_90', 'stock_mean_90', 'mean_in_stock_90', 'age', 'warehouse', 'category']


In [4]:
# ЭТАП 3.2. Обучение LightGBM по трём окнам и сравнение с базовыми прогнозами
import lightgbm as lgb

SEED = 42
N_JOBS = 4          # число потоков; для одинаковых результатов не меняй между запусками
WEIGHTS = {"SMAPE": 0.50, "WAPE": 0.20, "RMSE": 0.15, "Bias": 0.10, "RMSLE": 0.05}


def train_rate_model(X, y):
    """Модель предсказывает log1p(средние продажи в день за 64 дня)."""
    model = lgb.LGBMRegressor(
        n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=SEED, deterministic=True, force_row_wise=True,
        n_jobs=N_JOBS, verbose=-1,
    )
    model.fit(X, np.log1p(y))
    return model


def proxy_score(m, naive):
    """Ориентир: метрики, делённые на метрики «последнего значения» (как min-max в ТЗ), с весами."""
    return sum(WEIGHTS[k] * min(m[k] / naive[k], 1.0) for k in WEIGHTS)


THRESHOLDS = [0.0, 0.03, 0.05, 0.08, 0.12, 0.16, 0.20]
SCALES = [0.9, 1.0, 1.1, 1.2]

preds, naive_m, truths = {}, {}, {}
t0 = time.time()
for k in range(N_WINDOWS):
    c = window_cut(k)
    # учим только на датах, цель которых целиком заканчивается не позже дня c
    X_tr, y_tr = make_training_set(last_cut=c - HORIZON)
    model = train_rate_model(X_tr, y_tr)
    rate = np.clip(np.expm1(model.predict(build_features(c))), 0, None)
    preds[k], truths[k] = rate, truth(c)
    naive_m[k] = compute_metrics(truths[k], flat(Q0[:, c]))
    print(f"окно {k}: обучающих строк {len(X_tr):,}, готово за {time.time() - t0:.0f} c")
    if k == 0:
        imp = pd.Series(model.feature_importances_, X_tr.columns).sort_values(ascending=False)
        print("  самые важные признаки:", list(imp.index[:8]))


def evaluate(th, scale):
    rows = []
    for k in range(N_WINDOWS):
        r = preds[k] * scale
        r = np.where(r < th, 0.0, r)
        m = compute_metrics(truths[k], flat(r))
        rows.append({**m, "score": proxy_score(m, naive_m[k])})
    return pd.DataFrame(rows).mean()


grid = []
for th in THRESHOLDS:
    for sc in SCALES:
        grid.append({"порог": th, "масштаб": sc, **evaluate(th, sc)})
grid = pd.DataFrame(grid).sort_values("score")

print("\nМодель без обработки (порог 0, масштаб 1.0):")
display(grid[(grid["порог"] == 0.0) & (grid["масштаб"] == 1.0)].round(3))
print("Пять лучших сочетаний порога и масштаба по ориентиру score (меньше — лучше):")
display(grid.head(5).round(3))

# для сравнения: те же три окна и тот же ориентир для базовых прогнозов
base_rows = []
for name in baselines(window_cut(0)).keys():
    per_window = []
    for k in range(N_WINDOWS):
        m = compute_metrics(truths[k], baselines(window_cut(k))[name])
        per_window.append({**m, "score": proxy_score(m, naive_m[k])})
    base_rows.append({"подход": name, **pd.DataFrame(per_window).mean()})
print("\nБазовые прогнозы по тому же ориентиру:")
display(pd.DataFrame(base_rows).set_index("подход").round(3).sort_values("score"))

окно 0: обучающих строк 826,166, готово за 26 c
  самые важные признаки: ['category', 'share_pos_180', 'stock_mean_90', 'mean_180', 'warehouse', 'oos_share_90', 'mean_in_stock_90', 'mean_56']
окно 1: обучающих строк 747,065, готово за 50 c
окно 2: обучающих строк 667,964, готово за 72 c

Модель без обработки (порог 0, масштаб 1.0):


,порог,масштаб,SMAPE,WAPE,RMSE,Bias,RMSLE,score
1,0.0,1.0,181.88,99.188,0.784,66.517,0.304,0.868


Пять лучших сочетаний порога и масштаба по ориентиру score (меньше — лучше):


,порог,масштаб,SMAPE,WAPE,RMSE,Bias,RMSLE,score
24,0.20,0.9,162.383,84.828,0.787,79.737,0.302,0.834
20,0.16,0.9,161.998,86.625,0.786,77.611,0.301,0.835
16,0.12,0.9,161.762,88.690,0.785,75.676,0.300,0.837
25,0.20,1.0,161.535,87.412,0.787,80.257,0.306,0.837
21,0.16,1.0,161.174,89.336,0.786,79.194,0.305,0.839



Базовые прогнозы по тому же ориентиру:


,SMAPE,WAPE,RMSE,Bias,RMSLE,score
подход,,,,,,
"среднее 90 дн., ноль если < 0.1",161.742,91.435,0.785,77.585,0.303,0.843
медиана 28 дн.,168.121,76.650,0.825,92.646,0.326,0.850
среднее 90 дн.,176.720,96.488,0.785,64.227,0.302,0.862
среднее 28 дн.,168.955,94.772,0.782,79.247,0.304,0.867
среднее 7 дн.,164.436,98.684,0.828,124.727,0.326,0.899
нули,178.261,100.000,1.243,100.000,0.443,0.937
последнее значение,167.408,110.145,1.035,180.851,0.405,1.000


In [5]:
# ЭТАП 4.1. Сравнение вариантов модели и честный подбор порога
def make_training_set_w(last_cut, step=7, min_age=28):
    """То же, что make_training_set, но ещё возвращает «давность» каждой строки (в днях)."""
    parts_X, parts_y, parts_gap = [], [], []
    for cc in range(last_cut, 179, -step):
        X = build_features(cc)
        y = Q0[:, cc + 1:cc + 1 + HORIZON].mean(axis=1)
        keep = (cc - first_idx) >= min_age
        parts_X.append(X[keep])
        parts_y.append(y[keep])
        parts_gap.append(np.full(int(keep.sum()), last_cut - cc))
    X_all = pd.concat(parts_X, ignore_index=True)
    for col in ("warehouse", "category"):
        X_all[col] = X_all[col].astype("category")
    return X_all, np.concatenate(parts_y), np.concatenate(parts_gap)


# --- готовим данные по трём окнам один раз (чтобы не пересчитывать для каждого варианта) ---
t0 = time.time()
data = {}
for k in range(N_WINDOWS):
    c = window_cut(k)
    X_tr, y_tr, gap = make_training_set_w(c - HORIZON)
    data[k] = {"c": c, "X_tr": X_tr, "y_tr": y_tr, "gap": gap, "X_te": build_features(c)}
print(f"Данные подготовлены за {time.time() - t0:.0f} c")

BASE_PARAMS = dict(
    n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, deterministic=True, force_row_wise=True, n_jobs=N_JOBS, verbose=-1,
)


def fit_predict(d, drop=(), half_life=None, objective="log1p"):
    """Обучает модель на данных окна d и возвращает прогноз интенсивности по сериям."""
    X_tr = d["X_tr"].drop(columns=list(drop))
    X_te = d["X_te"].drop(columns=list(drop))
    w = None if half_life is None else 0.5 ** (d["gap"] / half_life)   # свежие даты весят больше
    if objective == "log1p":
        m = lgb.LGBMRegressor(**BASE_PARAMS).fit(X_tr, np.log1p(d["y_tr"]), sample_weight=w)
        r = np.expm1(m.predict(X_te))
    else:  # Tweedie: подходит для счётных данных с большим числом нулей
        m = lgb.LGBMRegressor(objective="tweedie", tweedie_variance_power=1.3, **BASE_PARAMS)
        m.fit(X_tr, d["y_tr"], sample_weight=w)
        r = m.predict(X_te)
    return np.clip(r, 0, None)


VARIANTS = {
    "база (log1p, все признаки)":      dict(),
    "без category":                    dict(drop=("category",)),
    "свежие даты важнее":              dict(half_life=180),
    "без category + свежие даты":      dict(drop=("category",), half_life=180),
    "Tweedie":                         dict(objective="tweedie"),
}

THS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
SCS = [0.8, 0.9, 1.0, 1.1]
combos = [(th, sc) for th in THS for sc in SCS]


def scores_for(rate, k):
    """Ориентир score для всех сочетаний (порог, масштаб) на окне k."""
    out = {}
    for th, sc in combos:
        r = rate * sc
        r = np.where(r < th, 0.0, r)
        out[(th, sc)] = proxy_score(compute_metrics(truths[k], flat(r)), naive_m[k])
    return out


rows, variant_rates = [], {}
for name, kw in VARIANTS.items():
    t1 = time.time()
    rates = {k: fit_predict(data[k], **kw) for k in range(N_WINDOWS)}
    variant_rates[name] = rates
    table = {k: scores_for(rates[k], k) for k in range(N_WINDOWS)}

    # честная оценка: порог выбираем на двух окнах, проверяем на третьем
    honest = []
    for held in range(N_WINDOWS):
        others = [k for k in range(N_WINDOWS) if k != held]
        best = min(combos, key=lambda cb: np.mean([table[k][cb] for k in others]))
        honest.append(table[held][best])
    # оптимистичная оценка: лучший порог по всем трём окнам сразу
    best_all = min(combos, key=lambda cb: np.mean([table[k][cb] for k in range(N_WINDOWS)]))
    rows.append({
        "вариант": name,
        "score честный": np.mean(honest),
        "score лучший (оптимистичный)": np.mean([table[k][best_all] for k in range(N_WINDOWS)]),
        "порог": best_all[0], "масштаб": best_all[1],
    })
    print(f"{name}: готово за {time.time() - t1:.0f} c")

result4 = pd.DataFrame(rows).set_index("вариант").round(4).sort_values("score честный")
print("\nСравнение вариантов (score: меньше — лучше; ориентир для простого базового прогноза ≈ 0.843):")
display(result4)

Данные подготовлены за 21 c
база (log1p, все признаки): готово за 55 c
без category: готово за 57 c
свежие даты важнее: готово за 56 c
без category + свежие даты: готово за 56 c
Tweedie: готово за 81 c

Сравнение вариантов (score: меньше — лучше; ориентир для простого базового прогноза ≈ 0.843):


,score честный,score лучший (оптимистичный),порог,масштаб
вариант,,,,
Tweedie,0.8387,0.8338,0.25,0.9
"база (log1p, все признаки)",0.8394,0.8343,0.25,0.9
свежие даты важнее,0.8394,0.8345,0.25,0.9
без category,0.8396,0.8344,0.20,0.9
без category + свежие даты,0.8396,0.8344,0.20,0.9


In [6]:
# ЭТАП 4.2. Добавляем недельную форму (день недели) к лучшему варианту
best_name = result4.index[0]
best_th, best_sc = float(result4.loc[best_name, "порог"]), float(result4.loc[best_name, "масштаб"])
print(f"Лучший вариант: {best_name}, порог {best_th}, масштаб {best_sc}")


def dow_profile(c):
    """Относительная сила каждого дня недели по последним 180 дням (среднее = 1)."""
    idx = np.arange(c - 179, c + 1)
    weekday = dates[idx].dayofweek.values
    daily_total = Q0[:, idx].sum(axis=0)
    prof = np.array([daily_total[weekday == d].mean() for d in range(7)])
    return prof / prof.mean()


def forecast_matrix(rate, c, use_dow):
    """Превращает интенсивность по сериям в матрицу прогноза (серии × 64 дня)."""
    r = rate * best_sc
    r = np.where(r < best_th, 0.0, r)
    P = flat(r)
    if use_dow:
        future_wd = (dates[c].dayofweek + 1 + np.arange(HORIZON)) % 7
        P = P * dow_profile(c)[future_wd][None, :]
    return P


out = []
for use_dow in (False, True):
    per_window = []
    for k in range(N_WINDOWS):
        c = window_cut(k)
        m = compute_metrics(truths[k], forecast_matrix(variant_rates[best_name][k], c, use_dow))
        per_window.append({**m, "score": proxy_score(m, naive_m[k])})
    out.append({"недельная форма": "да" if use_dow else "нет", **pd.DataFrame(per_window).mean()})

print("\nЭффект недельной формы:")
display(pd.DataFrame(out).set_index("недельная форма").round(3))
print("Профиль дней недели по последним 180 дням (пн … вс):",
      np.round(dow_profile(window_cut(0)), 3))

Лучший вариант: Tweedie, порог 0.25, масштаб 0.9

Эффект недельной формы:


,SMAPE,WAPE,RMSE,Bias,RMSLE,score
недельная форма,,,,,,
нет,162.762,83.169,0.787,81.893,0.304,0.834
да,162.769,83.184,0.787,81.891,0.304,0.834


Профиль дней недели по последним 180 дням (пн … вс): [0.996 0.98  1.015 0.961 0.952 1.003 1.092]


In [7]:
# ЭТАП 5. Сезонное окно: проверяем модель на августе–октябре прошлого года
SEASON_DATE = pd.Timestamp("2025-07-31")                # последний известный день окна
c_s = int((SEASON_DATE - dates[0]).days)
Y_s = truth(c_s)
naive_s = compute_metrics(Y_s, flat(Q0[:, c_s]))
print(f"Сезонное окно: известно до {dates[c_s].date()}, "
      f"факт {dates[c_s + 1].date()} — {dates[c_s + HORIZON].date()}")

# обучаем на датах, цель которых закончилась не позже c_s (будущее не подглядываем)
t0 = time.time()
X_s, y_s, gap_s = make_training_set_w(c_s - HORIZON)
model_s = lgb.LGBMRegressor(objective="tweedie", tweedie_variance_power=1.3, **BASE_PARAMS)
model_s.fit(X_s, y_s)
rate_s = np.clip(model_s.predict(build_features(c_s)), 0, None)
print(f"Обучено на {len(X_s):,} строках за {time.time() - t0:.0f} c")

# --- 1. Прогноз против факта по объёму: нет ли систематического занижения? ---
rows = []
for k in range(N_WINDOWS):
    c = window_cut(k)
    rows.append({"окно": f"проверка {k}", "до даты": dates[c].date(),
                 "прогноз / факт (объём)": variant_rates["Tweedie"][k].sum() / truths[k].mean(axis=1).sum()})
rows.append({"окно": "сезонное (авг–окт 2025)", "до даты": dates[c_s].date(),
             "прогноз / факт (объём)": rate_s.sum() / Y_s.mean(axis=1).sum()})
print("\nСумма прогноза к сумме факта (без порога и масштаба; 1.0 = в точку):")
display(pd.DataFrame(rows).round(3))

# --- 2. Какой масштаб и порог лучше всего на сезонном окне ---
SC_S = [0.9, 1.0, 1.1, 1.2, 1.3]
TH_S = [0.0, 0.10, 0.20, 0.25, 0.30]
grid_s = pd.DataFrame(index=TH_S, columns=SC_S, dtype=float)
for th in TH_S:
    for sc in SC_S:
        r = rate_s * sc
        r = np.where(r < th, 0.0, r)
        grid_s.loc[th, sc] = proxy_score(compute_metrics(Y_s, flat(r)), naive_s)
grid_s.index.name, grid_s.columns.name = "порог", "масштаб"
print("\nscore на сезонном окне по сочетаниям порога (строки) и масштаба (столбцы), меньше — лучше:")
display(grid_s.round(4))

best_th_s, best_sc_s = min(((th, sc) for th in TH_S for sc in SC_S), key=lambda x: grid_s.loc[x[0], x[1]])
m_best = compute_metrics(Y_s, flat(np.where(rate_s * best_sc_s < best_th_s, 0.0, rate_s * best_sc_s)))
m_old = compute_metrics(Y_s, flat(np.where(rate_s * 0.9 < 0.25, 0.0, rate_s * 0.9)))
print(f"\nЛучшее на сезонном окне: порог {best_th_s}, масштаб {best_sc_s}, score {grid_s.loc[best_th_s, best_sc_s]:.4f}")
print(f"Прежний выбор (порог 0.25, масштаб 0.9): score {grid_s.loc[0.25, 0.9]:.4f}")
display(pd.DataFrame({"лучшее": m_best, "прежнее": m_old}).T.round(3))

Сезонное окно: известно до 2025-07-31, факт 2025-08-01 — 2025-10-03
Обучено на 448,239 строках за 18 c

Сумма прогноза к сумме факта (без порога и масштаба; 1.0 = в точку):


,окно,до даты,прогноз / факт (объём)
0,проверка 0,2026-05-28,0.907
1,проверка 1,2026-03-25,1.042
2,проверка 2,2026-01-20,1.144
3,сезонное (авг–окт 2025),2025-07-31,0.828



score на сезонном окне по сочетаниям порога (строки) и масштаба (столбцы), меньше — лучше:


масштаб,0.9,1.0,1.1,1.2,1.3
порог,,,,,
0.00,0.8833,0.8856,0.8913,0.9000,0.9100
0.10,0.8704,0.8685,0.8702,0.8751,0.8837
0.20,0.8693,0.8656,0.8650,0.8678,0.8738
0.25,0.8698,0.8654,0.8646,0.8670,0.8725
0.30,0.8712,0.8661,0.8646,0.8662,0.8714



Лучшее на сезонном окне: порог 0.3, масштаб 1.1, score 0.8646
Прежний выбор (порог 0.25, масштаб 0.9): score 0.8698


,SMAPE,WAPE,RMSE,Bias,RMSLE
лучшее,164.231,83.940,0.797,86.441,0.289
прежнее,165.250,83.201,0.822,86.300,0.287


In [10]:
# ЭТАП 6. Финальное обучение, сборка submission.csv и проверки формата
import lightgbm as lgb

SEED = 42
N_JOBS = 4                 # не меняй число потоков между запусками, иначе возможны мелкие расхождения
FINAL_THRESHOLD = 0.25     # прогноз ниже порога превращается в 0
FINAL_SCALE = 1.0          # множитель на прогноз (компромисс между обычными окнами и сезонным)
CHECK_DETERMINISM = True   # обучить дважды и убедиться, что результат одинаковый

FINAL_PARAMS = dict(
    objective="tweedie", tweedie_variance_power=1.3,
    n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, deterministic=True, force_row_wise=True, n_jobs=N_JOBS, verbose=-1,
)

# --- 1. Обучение на всей доступной истории ---
# цель каждой строки лежит целиком в известном прошлом: последняя дата отсечки = D-1-64
t0 = time.time()
c_final = D - 1                                   # последний известный день (31.07.2026)
X_fin, y_fin = make_training_set(last_cut=c_final - HORIZON)
X_now = build_features(c_final)                   # признаки «на сегодня» для прогноза
print(f"Обучающая выборка: {len(X_fin):,} строк, подготовка за {time.time() - t0:.0f} c")


def train_and_predict():
    model = lgb.LGBMRegressor(**FINAL_PARAMS).fit(X_fin, y_fin)
    return model, np.clip(model.predict(X_now), 0, None)


t1 = time.time()
final_model, rate_final = train_and_predict()
print(f"Модель обучена за {time.time() - t1:.0f} c")

if CHECK_DETERMINISM:
    _, rate_again = train_and_predict()
    same = np.array_equal(rate_final, rate_again)
    print("Детерминизм (два обучения дали одинаковый результат):", "ДА" if same else "НЕТ")
    if not same:
        print("  максимальное расхождение:", float(np.abs(rate_final - rate_again).max()))

# --- 2. Постобработка: масштаб и обнуление малых прогнозов ---
rate_post = rate_final * FINAL_SCALE
rate_post = np.where(rate_post < FINAL_THRESHOLD, 0.0, rate_post)
P = flat(rate_post)                                # матрица (серии × 64 дня)

# --- 3. Сборка таблицы submission ---
future_dates = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=HORIZON)
sub = pd.DataFrame({
    "date": np.tile(future_dates.strftime("%Y-%m-%d"), S),
    "warehouse_name": np.repeat(meta["warehouse_name"].astype(str).values, HORIZON),
    "article": np.repeat(meta["article"].astype(str).values, HORIZON),
    "quantity": np.round(P.reshape(-1), 4),
})


# --- 4. Проверки формата (логика validate_submission из score.py) ---
def check_submission(df):
    problems = []
    if set(["date", "warehouse_name", "article", "quantity"]) - set(df.columns):
        return ["нет нужных колонок"]
    if df["quantity"].isna().any():
        problems.append(f"NaN в quantity: {int(df['quantity'].isna().sum())}")
    if (df["quantity"] < 0).any():
        problems.append(f"отрицательные значения: {int((df['quantity'] < 0).sum())}")
    if len(df) != 11 * 799 * HORIZON:
        problems.append(f"строк {len(df)}, ожидалось {11 * 799 * HORIZON}")
    ud = np.sort(pd.to_datetime(df["date"]).unique())
    if len(ud) != HORIZON:
        problems.append(f"уникальных дат {len(ud)}, ожидалось {HORIZON}")
    elif (pd.Series(ud).diff().dropna() != pd.Timedelta(days=1)).any():
        problems.append("даты идут не подряд")
    combos = df.groupby(["warehouse_name", "article"]).ngroups
    if combos != 11 * 799:
        problems.append(f"комбинаций склад × товар {combos}, ожидалось {11 * 799}")
    if df.duplicated(["date", "warehouse_name", "article"]).any():
        problems.append("есть дубли по date + warehouse_name + article")
    return problems


problems = check_submission(sub)
print("\nПроверка формата:", "ВСЁ В ПОРЯДКЕ" if not problems else problems)

# --- 5. Смысловая проверка: правдоподобен ли объём прогноза ---
last64 = Q0[:, c_final - HORIZON + 1:c_final + 1].sum()
print(f"Прогноз за 64 дня: {P.sum():,.0f} шт.; факт предыдущих 64 дней: {last64:,.0f} шт.; "
      f"отношение {P.sum() / last64:.2f}")
print(f"Серий, которым прогноз обнулён: {(rate_post == 0).sum()} из {S} ({(rate_post == 0).mean():.0%})")
print(f"Даты прогноза: {future_dates[0].date()} — {future_dates[-1].date()}")

# --- 6. Сохранение ---
if not problems:
    out_path = ROOT / "data" / "processed" / "submission.csv"
    sub.to_csv(out_path, index=False)
    print(f"\nФайл сохранён: {out_path} ({out_path.stat().st_size / 1e6:.1f} МБ)")
    display(sub.head(3))
else:
    print("\nФайл НЕ сохранён: исправь проблемы выше.")


Обучающая выборка: 905,267 строк, подготовка за 9 c
Модель обучена за 26 c
Детерминизм (два обучения дали одинаковый результат): ДА

Проверка формата: ВСЁ В ПОРЯДКЕ
Прогноз за 64 дня: 199,602 шт.; факт предыдущих 64 дней: 217,596 шт.; отношение 0.92
Серий, которым прогноз обнулён: 6274 из 8789 (71%)
Даты прогноза: 2026-08-01 — 2026-10-03

Файл сохранён: C:\Users\nikit\sber-forecast\data\processed\submission.csv (28.6 МБ)


,date,warehouse_name,article,quantity
0,2026-08-01,warehouse_name_0,char_article_0,0.0
1,2026-08-02,warehouse_name_0,char_article_0,0.0
2,2026-08-03,warehouse_name_0,char_article_0,0.0


In [9]:
# ЭТАП 7.1. Выносим код признаков и предсказания в src/ (единый источник кода для ноутбука и контейнера)
import sys

sys.path.insert(0, str(ROOT))
(ROOT / "src").mkdir(exist_ok=True)
(ROOT / "src" / "__init__.py").write_text("", encoding="utf-8")

FEATURES_PY = r'''"""Признаки для прогноза продаж. Единый источник кода для ноутбука и контейнера.

Правила: используются только данные до даты отсечки включительно, окна не длиннее 180 дней.
"""
import warnings

import numpy as np
import pandas as pd

WINDOW = 180
AGE_CAP = WINDOW - 1   # контейнер видит ровно 180 дней, поэтому возраст ряда ограничен 179

FEATURE_ORDER = [
    "mean_7", "mean_14", "mean_28", "mean_56", "mean_90", "mean_180",
    "share_pos_28", "share_pos_90", "share_pos_180", "max_90", "max_180",
    "days_since_sale", "stock_last", "stock_missing_last", "oos_share_30", "oos_share_90",
    "stock_missing_90", "stock_mean_90", "mean_in_stock_90", "age", "warehouse", "category",
]


def build_features(Q0, ST, c, first_idx, wh_code, cat_code):
    """
    Q0  — продажи (серии x дни), пропуски заменены нулями
    ST  — остатки (серии x дни), NaN = данных нет
    c   — индекс последнего известного дня
    first_idx — индекс первого дня ряда, wh_code / cat_code — коды склада и категории
    """
    f = {}
    for w in (7, 14, 28, 56, 90, 180):
        f[f"mean_{w}"] = Q0[:, c - w + 1:c + 1].mean(axis=1)
    for w in (28, 90, 180):
        f[f"share_pos_{w}"] = (Q0[:, c - w + 1:c + 1] > 0).mean(axis=1)
    for w in (90, 180):
        f[f"max_{w}"] = Q0[:, c - w + 1:c + 1].max(axis=1)

    pos = Q0[:, c - 179:c + 1] > 0
    f["days_since_sale"] = np.where(pos.any(axis=1), 179 - np.argmax(pos[:, ::-1], axis=1), 180)

    s = ST[:, c - 179:c + 1]
    f["stock_last"] = s[:, -1]
    f["stock_missing_last"] = np.isnan(s[:, -1]).astype(np.float32)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for w in (30, 90):
            sw = s[:, -w:]
            zero_share = np.where(np.isnan(sw), np.nan, (sw == 0).astype(np.float32))
            f[f"oos_share_{w}"] = np.nanmean(zero_share, axis=1)
        f["stock_missing_90"] = np.isnan(s[:, -90:]).mean(axis=1)
        f["stock_mean_90"] = np.nanmean(s[:, -90:], axis=1)

    q90 = Q0[:, c - 89:c + 1]
    in_stock = ST[:, c - 90:c] > 0
    cnt_in = in_stock.sum(axis=1)
    f["mean_in_stock_90"] = np.where(
        cnt_in > 0, (q90 * in_stock).sum(axis=1) / np.maximum(cnt_in, 1), np.nan)

    f["age"] = np.minimum(c - first_idx, AGE_CAP)
    f["warehouse"] = wh_code
    f["category"] = cat_code

    X = pd.DataFrame(f)[FEATURE_ORDER]
    X["warehouse"] = X["warehouse"].astype("category")
    X["category"] = X["category"].astype("category")
    return X
'''

PREDICTOR_PY = r'''"""Предсказание по массивам последних 180 дней. Это ядро для predictor.py в контейнере.

Контракт: на вход матрицы продаж и остатков (серии x 180 дней, NaN = данных нет),
названия складов и артикулов каждой серии. На выходе матрица прогноза (серии x 64 дня).
"""
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np

from . import features


def load_artifacts(model_dir):
    model_dir = Path(model_dir)
    cfg = json.loads((model_dir / "config.json").read_text(encoding="utf-8"))
    booster = lgb.Booster(model_file=str(model_dir / "model.txt"))
    return {"cfg": cfg, "booster": booster}


def predict_from_arrays(Qw, STw, warehouses, articles, art):
    cfg = art["cfg"]
    window = cfg["window"]
    if Qw.shape[1] != window or STw.shape[1] != window:
        raise ValueError(f"Нужно ровно {window} дней, получено {Qw.shape[1]}")

    wh_index = {n: i for i, n in enumerate(cfg["warehouses"])}
    cat_index = {n: i for i, n in enumerate(cfg["categories"])}
    unknown = cat_index.get("unknown", -1)
    wh_code = np.array([wh_index.get(w, -1) for w in warehouses])
    cat_code = np.array([cat_index.get(cfg["article_category"].get(a, "unknown"), unknown)
                         for a in articles])

    Q0 = np.nan_to_num(Qw)
    first_idx = np.argmax(~np.isnan(Qw), axis=1)
    X = features.build_features(Q0, STw, window - 1, first_idx, wh_code, cat_code)

    rate = np.clip(art["booster"].predict(X), 0, None) * cfg["scale"]
    rate = np.where(rate < cfg["threshold"], 0.0, rate)
    return np.repeat(rate[:, None], cfg["horizon"], axis=1)
'''

(ROOT / "src" / "features.py").write_text(FEATURES_PY, encoding="utf-8")
(ROOT / "src" / "predictor_core.py").write_text(PREDICTOR_PY, encoding="utf-8")

# --- подключаем вынесенный код вместо старой версии из ноутбука ---
import importlib
from src import features as _features
importlib.reload(_features)

X_old = build_features(D - 1)          # старая версия (из этапа 3.1) — для сравнения


def build_features(c):
    """Обёртка: те же признаки, но из src/features.py (то, что будет работать в контейнере)."""
    return _features.build_features(Q0, ST, c, first_idx, wh_code, cat_code)


X_new = build_features(D - 1)
changed = [col for col in _features.FEATURE_ORDER if not np.allclose(
    X_old[col].astype(float), X_new[col].astype(float), equal_nan=True)]
print("Файлы записаны: src/features.py, src/predictor_core.py")
print("Признаков:", X_new.shape[1], "| изменились относительно старой версии:", changed or "ни один")

Файлы записаны: src/features.py, src/predictor_core.py
Признаков: 22 | изменились относительно старой версии: ['age']


In [11]:
# ЭТАП 7.2. Сохраняем модель и справочники в model/ и проверяем, что контейнер воспроизведёт прогноз
import json

from src import predictor_core

model_dir = ROOT / "model"
model_dir.mkdir(exist_ok=True)

# --- 1. Сохранение модели и справочников ---
final_model.booster_.save_model(str(model_dir / "model.txt"))
cfg = {
    "horizon": HORIZON,
    "window": 180,
    "threshold": FINAL_THRESHOLD,
    "scale": FINAL_SCALE,
    "warehouses": [str(x) for x in meta["warehouse_name"].cat.categories],   # порядок = коды складов
    "categories": [str(x) for x in pd.Categorical(meta["cat1"]).categories],  # порядок = коды категорий
    "article_category": {str(a): str(cat) for a, cat in art_cat.items()},
    "feature_order": list(_features.FEATURE_ORDER),
    "seed": SEED,
}
(model_dir / "config.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print("Сохранено в model/: model.txt", f"({(model_dir / 'model.txt').stat().st_size / 1e6:.1f} МБ),",
      "config.json", f"({(model_dir / 'config.json').stat().st_size / 1e3:.0f} КБ)")

# --- 2. Тест «как в контейнере»: видим только последние 180 дней, модель берём из файлов ---
art = predictor_core.load_artifacts(model_dir)
c = D - 1
P_core = predictor_core.predict_from_arrays(
    Q[:, c - 179:c + 1], ST[:, c - 179:c + 1],
    meta["warehouse_name"].astype(str).values, meta["article"].astype(str).values, art,
)
diff = float(np.abs(P_core - P).max())
same = np.allclose(P_core, P, rtol=1e-5, atol=1e-5)   # допуск на округление float32
print(f"\nМаксимальное расхождение с прогнозом из ноутбука: {diff:.2e}")
print("ПАРИТЕТ (180 дней + модель из файла = прогноз из ноутбука):", "ДА" if same else "НЕТ")
if not same:
    bad = ~np.isclose(P_core, P, rtol=1e-5, atol=1e-5).all(axis=1)
    print("Серий с расхождением:", int(bad.sum()), "из", S)

Сохранено в model/: model.txt (2.8 МБ), config.json (37 КБ)

Максимальное расхождение с прогнозом из ноутбука: 9.33e-07
ПАРИТЕТ (180 дней + модель из файла = прогноз из ноутбука): ДА
